In [ ]:
# ==============================================================================
# RANDOM FOREST AND IHT
# ==============================================================================

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

In [ ]:

import os
import gc
import json
import time
import warnings
from collections import Counter
from itertools import product

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.inspection import permutation_importance
from imblearn.under_sampling import InstanceHardnessThreshold

warnings.filterwarnings("default")

# Print messages immediately so long runs show progress in real time.

def log(msg):
    print(msg, flush=True)



In [3]:

# ============================================================================
# CONFIG
# ============================================================================


In [4]:

SEED = 42
RAW_DATA_PATH = os.path.join("..", "data", "processed", "dados_uteis_refreshed_v3_memorysafe.pkl")
OUTPUT_DIR = os.path.join("..", "data", "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COL = "ausencia"
RUN_TAG = "rf_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained_confusion"

DROP_COLS = [
    "cns_usuario", "ibge_usuario", "cep_usuario", "cep_cnes_exe",
    "ibge_cnes_exe", "lat_cnes", "lon_cnes", "lat_usuario", "lon_usuario",
    "cep_aprox_usuario", "cep_aprox_cnes", "dta_atend", "cod_uf_usuario",
    "cod_uf_cnes", "cidade_usuario", "cidade_cnes"
]



In [5]:

# Deina-style nested workflow
N_OUTER_FOLDS = 10  # z1
N_INNER_FOLDS = 10  # z2
TOP_K_FEATURES = 20


In [6]:

# Memory-safe caps for RF on local hardware
# Set to None if you really want the exact full-data outer calibration folds.
OUTER_CALIBRATION_CAP = 800_000
INNER_TRAIN_CAP = 250_000


In [7]:

# IHT only inside training partitions
IHT_SAMPLING_STRATEGY = 0.5
IHT_CV = 3
IHT_HARDNESS_MAX_ITER = 3000
IHT_HARDNESS_TOL = 1e-4
IHT_HARDNESS_ALPHA = 0.0001
IHT_HARDNESS_CLASS_WEIGHT = "balanced"


In [8]:

# Search on resampled train, score on raw inner validation
SEARCH_SIZE_CAP = 60_000
N_CANDIDATES_PER_INNER = 8


In [9]:

# Permutation importance for feature ranking (inner folds only)
PERM_RANK_MAX_SAMPLES = 10_000
PERM_RANK_REPEATS = 3


In [10]:

# Final performance emphasis
# We use Youden's J because the previous F1 threshold selection was unstable.
# The constraints below prevent degenerate all-positive / almost-all-positive behavior.
THRESHOLD_OBJECTIVE = "youden_j"  # options: "youden_j" or "f1"
MANUAL_THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
MIN_THRESHOLD = 0.10
MIN_SPECIFICITY_FOR_THRESHOLD = 0.20
MIN_PRECISION_MARGIN_OVER_PREVALENCE = 0.00
MAX_PREDICTED_POSITIVE_RATE = 0.70
MIN_PREDICTED_NEGATIVE_RATE = 0.05
USE_SIGMOID_PROBA_CALIBRATION = True
FINAL_PROBA_CALIBRATION_SIZE = 50_000
FINAL_THRESHOLD_SIZE = 50_000


In [11]:

# Optional final outer-best-fold permutation disabled by default
RUN_OUTER_BEST_FOLD_PERMUTATION = False
OUTER_PERM_MAX_SAMPLES = 30_000
OUTER_PERM_REPEATS = 3


In [12]:

# RF candidate space based on your most valid RF code
RF_PARAM_GRID = {
    "n_estimators": [200, 300, 400],
    "max_depth": [12, 16, 20],
    "min_samples_split": [5, 10, 20],
    "min_samples_leaf": [5, 10, 20],
    "max_features": ["sqrt", 0.3],
    "bootstrap": [True],
    "class_weight": [None, "balanced_subsample"],
}



In [ ]:
# ============================================================================
# HELPERS
# ============================================================================

In [ ]:
# Build a dense OneHotEncoder while supporting both old and new sklearn parameter names.

def make_dense_ohe(categories=None):
    kwargs = {"handle_unknown": "ignore"}
    if categories is not None:
        kwargs["categories"] = categories
    try:
        return OneHotEncoder(sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=False, **kwargs)



In [ ]:
# Coerce labels safely to integer 0/1 and fail early if labels are missing or not binary.

def ensure_binary_int_labels(y, name="y"):
    if isinstance(y, pd.Series):
        arr = y.to_numpy()
    else:
        arr = np.asarray(y)

    arr = arr.ravel()

    if pd.isna(arr).any():
        raise ValueError(f"{name} contains missing values.")

    try:
        arr_num = pd.to_numeric(pd.Series(arr), errors="raise").to_numpy()
    except Exception:
        uniques = pd.Series(arr).dropna().unique().tolist()
        if len(uniques) != 2:
            raise ValueError(f"{name} is not binary and could not be coerced safely.")
        mapping = {uniques[0]: 0, uniques[1]: 1}
        arr_num = pd.Series(arr).map(mapping).to_numpy()

    arr_int = arr_num.astype(np.int64)
    unique_vals = np.unique(arr_int)
    valid_sets = [np.array([0, 1]), np.array([0]), np.array([1])]
    if not any(np.array_equal(unique_vals, s) for s in valid_sets):
        raise ValueError(f"{name} must contain only 0/1 after coercion. Found: {unique_vals}")

    return arr_int



In [ ]:
# Return TN, FP, FN, TP with fixed 0/1 labels, even when one class is absent.

def safe_binary_confusion(y_true, y_pred):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_pred = ensure_binary_int_labels(y_pred, "y_pred")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape != (2, 2):
        padded = np.zeros((2, 2), dtype=int)
        padded[:cm.shape[0], :cm.shape[1]] = cm
        cm = padded
    tn, fp, fn, tp = cm.ravel()
    return tn, fp, fn, tp



In [ ]:
# Compute specificity, TN / (TN + FP), with a safe zero-denominator fallback.

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)
    denom = tn + fp
    return 0.0 if denom == 0 else tn / denom



In [ ]:
# Compute negative predictive value and return NaN when no predicted negatives exist.

def npv_score(y_true, y_pred):
    """Negative Predictive Value: TN / (TN + FN).

    If there are no predicted negatives, NPV is mathematically undefined.
    Returning np.nan is more honest than returning 0.0, because 0.0 would mean
    the model made negative predictions and all of them were wrong.
    """
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)
    denom = tn + fn
    return np.nan if denom == 0 else tn / denom


In [ ]:
# Compute ROC AUC, returning NaN instead of crashing when AUC is undefined.

def safe_roc_auc(y_true, y_proba):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    try:
        return roc_auc_score(y_true, y_proba)
    except ValueError:
        return np.nan


In [ ]:
# Collect fold metrics, confusion counts, class rates, and degenerate-prediction flags.

def calculate_metrics(y_true, y_pred, y_proba):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_pred = ensure_binary_int_labels(y_pred, "y_pred")
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)

    real_neg = tn + fp
    real_pos = tp + fn
    pred_neg = tn + fn
    pred_pos = tp + fp
    total = real_neg + real_pos

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": safe_roc_auc(y_true, y_proba),

        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),

        "Real_Positive": int(real_pos),
        "Real_Negative": int(real_neg),
        "Predicted_Positive": int(pred_pos),
        "Predicted_Negative": int(pred_neg),

        "Real_Positive_Rate": real_pos / total if total else np.nan,
        "Real_Negative_Rate": real_neg / total if total else np.nan,
        "Predicted_Positive_Rate": pred_pos / total if total else np.nan,
        "Predicted_Negative_Rate": pred_neg / total if total else np.nan,

        "Has_No_Predicted_Negatives": bool(pred_neg == 0),
        "Has_No_Predicted_Positives": bool(pred_pos == 0),
    }


In [ ]:
# Return class-0 and class-1 counts after enforcing binary labels.

def get_class_counts(y):
    y = ensure_binary_int_labels(y, "y_counts")
    counts = np.bincount(y, minlength=2)
    return int(counts[0]), int(counts[1])



In [ ]:
# Log total samples, class counts, percentages, and imbalance ratio.

def print_class_distribution(name, y):
    c0, c1 = get_class_counts(y)
    total = c0 + c1
    p0 = c0 / total * 100 if total else 0.0
    p1 = c1 / total * 100 if total else 0.0
    ratio = np.inf if min(c0, c1) == 0 else max(c0, c1) / min(c0, c1)
    log(f"\n{name}")
    log("-" * len(name))
    log(f"Total: {total:,}".replace(",", "."))
    log(f"Class 0: {c0:,} ({p0:.4f}%)".replace(",", "."))
    log(f"Class 1: {c1:,} ({p1:.4f}%)".replace(",", "."))
    if np.isinf(ratio):
        log("Imbalance ratio (maj/min): inf")
    else:
        log(f"Imbalance ratio (maj/min): {ratio:.4f}:1")



In [ ]:
# Format class counts and imbalance ratio as one compact log string.

def format_class_counts_inline(y):
    c0, c1 = get_class_counts(y)
    total = c0 + c1
    ratio = np.inf if min(c0, c1) == 0 else max(c0, c1) / min(c0, c1)
    ratio_text = "inf" if np.isinf(ratio) else f"{ratio:.4f}:1"
    return (
        f"total={total:,}, class0={c0:,}, class1={c1:,}, "
        f"maj/min={ratio_text}"
    ).replace(",", ".")



In [ ]:
# Check that manually supplied thresholds are inside (0, 1) and remove duplicates.

def validate_manual_thresholds(thresholds):
    valid = []
    for t in thresholds:
        t = float(t)
        if not (0.0 < t < 1.0):
            raise ValueError(f"Threshold {t} is invalid. Thresholds must be between 0 and 1.")
        valid.append(t)
    return sorted(set(valid))



In [ ]:
# Select the best probability threshold while blocking degenerate all-positive or no-negative solutions.

def optimize_threshold_constrained(
    y_true,
    y_proba,
    objective="youden_j",
    min_threshold=MIN_THRESHOLD,
    min_specificity=MIN_SPECIFICITY_FOR_THRESHOLD,
    min_precision=None,
    precision_margin_over_prevalence=MIN_PRECISION_MARGIN_OVER_PREVALENCE,
    max_predicted_positive_rate=MAX_PREDICTED_POSITIVE_RATE,
    min_predicted_negative_rate=MIN_PREDICTED_NEGATIVE_RATE,
):
    """Choose a threshold while preventing pathological positive-heavy solutions.

    Youden's J balances sensitivity and specificity. The constraints are a
    safety layer so a threshold cannot be selected if it predicts almost
    everyone as positive or leaves no negative predictions for NPV.
    """
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_proba = np.asarray(y_proba, dtype=np.float64)

    prevalence = float(np.mean(y_true))
    if min_precision is None:
        min_precision = prevalence + precision_margin_over_prevalence

    grid = np.linspace(min_threshold, 0.99, 90)
    quantiles = np.quantile(y_proba, np.linspace(0.01, 0.99, 99))
    manual = validate_manual_thresholds(MANUAL_THRESHOLDS)

    thresholds = np.unique(np.concatenate([grid, quantiles, manual]))
    thresholds = thresholds[(thresholds >= min_threshold) & (thresholds < 1.0)]

    rows = []
    for thr in thresholds:
        y_pred = (y_proba >= thr).astype(np.uint8)
        tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)

        total = len(y_true)
        pred_pos = tp + fp
        pred_neg = tn + fn
        pred_pos_rate = pred_pos / total if total else np.nan
        pred_neg_rate = pred_neg / total if total else np.nan

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        specificity = specificity_score(y_true, y_pred)
        npv = npv_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        youden_j = recall + specificity - 1.0
        balanced_accuracy = (recall + specificity) / 2.0

        passes_constraints = (
            specificity >= min_specificity
            and precision >= min_precision
            and pred_pos_rate <= max_predicted_positive_rate
            and pred_neg_rate >= min_predicted_negative_rate
        )

        rows.append({
            "Threshold": float(thr),
            "Precision": float(precision),
            "Recall": float(recall),
            "Specificity": float(specificity),
            "NPV": float(npv) if not pd.isna(npv) else np.nan,
            "F1": float(f1),
            "Youden_J": float(youden_j),
            "Balanced_Accuracy": float(balanced_accuracy),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp),
            "Predicted_Positive": int(pred_pos),
            "Predicted_Negative": int(pred_neg),
            "Predicted_Positive_Rate": float(pred_pos_rate),
            "Predicted_Negative_Rate": float(pred_neg_rate),
            "Min_Precision_Required": float(min_precision),
            "Min_Specificity_Required": float(min_specificity),
            "Max_Predicted_Positive_Rate_Allowed": float(max_predicted_positive_rate),
            "Min_Predicted_Negative_Rate_Required": float(min_predicted_negative_rate),
            "Passes_Constraints": bool(passes_constraints),
        })

    df_thr = pd.DataFrame(rows)

    if objective.lower() == "f1":
        score_col = "F1"
    elif objective.lower() == "youden_j":
        score_col = "Youden_J"
    else:
        raise ValueError("objective must be 'f1' or 'youden_j'")

    df_valid = df_thr[df_thr["Passes_Constraints"]].copy()
    threshold_selection_mode = "constrained"

    if len(df_valid) == 0:
        df_valid = df_thr[
            (df_thr["Predicted_Negative"] > 0)
            & (df_thr["Specificity"] > 0)
        ].copy()
        threshold_selection_mode = "fallback_non_degenerate"

    if len(df_valid) == 0:
        df_valid = df_thr.copy()
        threshold_selection_mode = "fallback_all_thresholds"

    best_row = (
        df_valid.sort_values(
            by=[score_col, "Specificity", "Precision", "Recall", "Threshold"],
            ascending=[False, False, False, False, True],
        )
        .iloc[0]
        .to_dict()
    )
    best_row["Threshold_Selection_Mode"] = threshold_selection_mode
    best_row["Threshold_Valid_Candidates"] = int(len(df_valid))
    best_row["Threshold_Total_Candidates"] = int(len(df_thr))

    return best_row, df_thr



In [ ]:
# Clip probabilities away from 0 and 1 so logit calibration stays numerically stable.

def clip_proba(p, eps=1e-6):
    return np.clip(np.asarray(p, dtype=np.float64), eps, 1.0 - eps)



In [ ]:
# Convert probabilities to log-odds for sigmoid/Platt-style calibration.

def logit(p):
    p = clip_proba(p)
    return np.log(p / (1.0 - p))



In [ ]:
# Fit calibration on natural-distribution data to correct probability shift caused by IHT.

def fit_sigmoid_calibrator(y_calib, proba_calib, seed=SEED):
    """Fit a one-dimensional sigmoid/Platt-style calibrator.

    It is trained on natural-distribution data, not IHT-resampled data. This
    helps correct probability shift caused by training on an artificially
    rebalanced class distribution.
    """
    y_calib = ensure_binary_int_labels(y_calib, "y_calib")
    if len(np.unique(y_calib)) < 2:
        return None

    z = logit(proba_calib).reshape(-1, 1)
    calibrator = LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=seed,
    )
    calibrator.fit(z, y_calib)
    return calibrator



In [ ]:
# Apply the fitted probability calibrator, or return raw probabilities if calibration was skipped.

def apply_sigmoid_calibrator(calibrator, proba):
    if calibrator is None:
        return np.asarray(proba, dtype=np.float64)
    z = logit(proba).reshape(-1, 1)
    return calibrator.predict_proba(z)[:, 1]


In [ ]:
# Split outer-training data into final-fit, calibration, and threshold sets without touching outer validation.

def split_final_train_calib_threshold_indices(
    y,
    calib_size=FINAL_PROBA_CALIBRATION_SIZE,
    threshold_size=FINAL_THRESHOLD_SIZE,
    seed=SEED,
):
    """Split outer calibration into final-train, probability-calibration, threshold sets.

    All three partitions remain inside the outer calibration side. The final
    validation fold is never used for calibration or threshold tuning.
    """
    y = ensure_binary_int_labels(y, "y_final_split")
    idx_all = np.arange(len(y))

    requested_holdout = int(calib_size or 0) + int(threshold_size or 0)
    if requested_holdout <= 0 or len(y) < 10:
        return idx_all, np.array([], dtype=int), np.array([], dtype=int)

    # Keep at least half the capped calibration side for final model fitting.
    max_holdout = max(2, len(y) // 2)
    total_holdout = min(requested_holdout, max_holdout)

    idx_train, idx_temp = train_test_split(
        idx_all,
        test_size=total_holdout,
        stratify=y,
        random_state=seed,
    )

    if len(idx_temp) < 4:
        return idx_train, idx_temp, idx_temp

    threshold_n = min(int(threshold_size or 0), len(idx_temp) - 2)
    if threshold_n <= 0:
        return idx_train, idx_temp, idx_temp

    idx_calib, idx_threshold = train_test_split(
        idx_temp,
        test_size=threshold_n,
        stratify=y[idx_temp],
        random_state=seed + 1,
    )
    return idx_train, idx_calib, idx_threshold



In [ ]:
# Learn numeric columns, categorical columns, and category levels from the outer training fold only.

def build_outer_schema(X_outer_raw):
    num_feats = X_outer_raw.select_dtypes(include=["number"]).columns.tolist()
    cat_feats = X_outer_raw.select_dtypes(include=["category", "object"]).columns.tolist()
    cat_categories = [pd.Index(X_outer_raw[col].dropna().unique()).tolist() for col in cat_feats]
    return num_feats, cat_feats, cat_categories



In [ ]:
# Build MinMax scaling plus fixed-category one-hot preprocessing to avoid leakage.

def build_preprocessor_from_schema(num_feats, cat_feats, cat_categories):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", MinMaxScaler(), num_feats),
            ("cat", make_dense_ohe(categories=cat_categories), cat_feats),
        ],
        verbose_feature_names_out=False,
    )
    return preprocessor



In [ ]:
# Create a stratified capped subset to keep large operations memory-safe.

def make_cap_subset_indices(y, cap, seed):
    y = ensure_binary_int_labels(y, "y_cap")
    if cap is None or len(y) <= cap:
        return np.arange(len(y))
    idx = np.arange(len(y))
    _, idx_small = train_test_split(
        idx,
        test_size=cap,
        stratify=y,
        random_state=seed,
    )
    return idx_small



In [ ]:
# Build a stratified smaller training subset for faster hyperparameter search.

def make_search_subset(X_train, y_train, max_size, seed):
    y_train = ensure_binary_int_labels(y_train, "y_search")
    if len(X_train) <= max_size:
        return X_train, y_train
    indices = np.arange(len(X_train))
    _, idx_small = train_test_split(
        indices,
        test_size=max_size,
        stratify=y_train,
        random_state=seed,
    )
    return X_train[idx_small], y_train[idx_small]



In [ ]:
# Build a stratified smaller validation subset for faster permutation importance.

def make_perm_subset(X, y, max_size, seed):
    y = ensure_binary_int_labels(y, "y_perm_subset")
    if len(X) <= max_size:
        return X, y
    indices = np.arange(len(X))
    _, idx_small = train_test_split(
        indices,
        test_size=max_size,
        stratify=y,
        random_state=seed,
    )
    return X[idx_small], y[idx_small]



In [ ]:
# Apply Instance Hardness Threshold undersampling only on training data and log the before/after balance.

def apply_iht(X_train, y_train, seed):
    y_train = ensure_binary_int_labels(y_train, "y_train_for_iht")
    hardness_estimator = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=IHT_HARDNESS_ALPHA,
        max_iter=IHT_HARDNESS_MAX_ITER,
        tol=IHT_HARDNESS_TOL,
        class_weight=IHT_HARDNESS_CLASS_WEIGHT,
        random_state=seed,
        n_jobs=1,
    )
    iht = InstanceHardnessThreshold(
        estimator=hardness_estimator,
        sampling_strategy=IHT_SAMPLING_STRATEGY,
        cv=IHT_CV,
        random_state=seed,
        n_jobs=1,
    )
    start = time.time()
    X_res, y_res = iht.fit_resample(X_train, y_train)
    elapsed_min = (time.time() - start) / 60.0
    X_res = np.asarray(X_res, dtype=np.float32)
    y_res = ensure_binary_int_labels(y_res, "y_resampled_iht")
    log(
        "IHT distribution | "
        f"before: {format_class_counts_inline(y_train)} | "
        f"after: {format_class_counts_inline(y_res)} | "
        f"sampling_strategy={IHT_SAMPLING_STRATEGY}"
    )
    return X_res, y_res, iht, elapsed_min



In [ ]:
# Expand the full random-forest hyperparameter grid into candidate dictionaries.

def build_candidate_pool():
    keys = list(RF_PARAM_GRID.keys())
    values = [RF_PARAM_GRID[k] for k in keys]
    rows = []
    for combo in product(*values):
        row = dict(zip(keys, combo))
        row["Candidate_Key"] = (
            f"n_estimators={row['n_estimators']}|"
            f"max_depth={row['max_depth']}|"
            f"min_samples_split={row['min_samples_split']}|"
            f"min_samples_leaf={row['min_samples_leaf']}|"
            f"max_features={row['max_features']}|"
            f"bootstrap={row['bootstrap']}|"
            f"class_weight={row['class_weight']}"
        )
        rows.append(row)
    return rows



In [ ]:
# Randomly sample a limited number of RF candidates to reduce inner-loop cost.

ALL_RF_CANDIDATES = build_candidate_pool()

def sample_candidate_list(seed):
    rng = np.random.default_rng(seed)
    n_take = min(N_CANDIDATES_PER_INNER, len(ALL_RF_CANDIDATES))
    idx = rng.choice(len(ALL_RF_CANDIDATES), size=n_take, replace=False)
    return [ALL_RF_CANDIDATES[i] for i in idx]



In [ ]:
# Create a RandomForestClassifier from one sampled candidate parameter set.

def fit_rf_from_params(params, seed):
    return RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        bootstrap=params["bootstrap"],
        class_weight=params["class_weight"],
        random_state=seed,
        n_jobs=1,
    )


In [ ]:
# Train sampled RF candidates and choose the one with best validation AUC.

def evaluate_candidate_grid_on_validation(X_search, y_search, X_val, y_val, seed):
    candidate_list = sample_candidate_list(seed)
    rows = []
    best_score = -np.inf
    best_params = None

    for params in candidate_list:
        model = fit_rf_from_params(params, seed)
        model.fit(X_search, y_search)
        y_val_proba = model.predict_proba(X_val)[:, 1]
        val_auc = safe_roc_auc(y_val, y_val_proba)
        rows.append({
            "Candidate_Key": params["Candidate_Key"],
            **{k: params[k] for k in RF_PARAM_GRID.keys()},
            "Validation_AUC": val_auc,
        })
        if np.isfinite(val_auc) and val_auc > best_score:
            best_score = val_auc
            best_params = {k: params[k] for k in RF_PARAM_GRID.keys()}

    if best_params is None:
        raise RuntimeError("No valid RF candidate produced a valid AUC score.")

    return best_params, best_score, pd.DataFrame(rows)



In [ ]:
# Scorer used by permutation importance, based on predicted class-1 probabilities.

def roc_auc_proba_scorer(estimator, X, y):
    y = ensure_binary_int_labels(y, "y_score")
    y_proba = estimator.predict_proba(X)[:, 1]
    return roc_auc_score(y, y_proba)



In [ ]:
# Compute AUC-based permutation importance and return both aligned vector and ranked table.

def compute_permutation_importance_vector(model, X_val, y_val, feature_names, seed):
    X_perm, y_perm = make_perm_subset(X_val, y_val, PERM_RANK_MAX_SAMPLES, seed)
    perm_results = permutation_importance(
        estimator=model,
        X=X_perm,
        y=y_perm,
        scoring=roc_auc_proba_scorer,
        n_repeats=PERM_RANK_REPEATS,
        random_state=seed,
        n_jobs=1,
        max_samples=1.0,
    )
    df_perm = pd.DataFrame({
        "Feature": feature_names,
        "Importance_Mean": perm_results.importances_mean,
        "Importance_Std": perm_results.importances_std,
    }).sort_values(by="Importance_Mean", ascending=False).reset_index(drop=True)
    importance_vector = (
        df_perm.set_index("Feature")["Importance_Mean"]
        .reindex(feature_names)
        .fillna(0.0)
        .to_numpy()
    )
    return importance_vector, df_perm



In [ ]:
# Summarize outer-fold metrics with mean, standard deviation, min, max, and missing counts.

def summarize_outer_results(df_outer):
    metric_cols = [
        "Accuracy", "Precision", "Recall", "Specificity", "NPV", "F1", "ROC_AUC",
        "Threshold", "Selected_Threshold", "Inner_Median_Threshold",
        "Predicted_Positive_Rate", "Predicted_Negative_Rate",
        "Inner_Best_AUC_Mean", "Inner_IHT_Time_Min_Mean", "Outer_IHT_Time_Min",
    ]
    rows = []
    for col in metric_cols:
        if col in df_outer.columns:
            rows.append({
                "Metric": col,
                "Mean": df_outer[col].mean(skipna=True),
                "Std": df_outer[col].std(ddof=1, skipna=True),
                "Min": df_outer[col].min(skipna=True),
                "Max": df_outer[col].max(skipna=True),
                "Valid_Folds": int(df_outer[col].notna().sum()),
                "Missing_or_Undefined_Folds": int(df_outer[col].isna().sum()),
            })
    return pd.DataFrame(rows)



In [ ]:
# Pool TP, TN, FP, and FN across folds and recompute global confusion-derived metrics.
    
def summarize_pooled_confusion(df_outer):
    required = ["TP", "TN", "FP", "FN"]
    if not all(c in df_outer.columns for c in required):
        return pd.DataFrame()

    total_tp = int(df_outer["TP"].sum())
    total_tn = int(df_outer["TN"].sum())
    total_fp = int(df_outer["FP"].sum())
    total_fn = int(df_outer["FN"].sum())
    total = total_tp + total_tn + total_fp + total_fn

    accuracy = (total_tp + total_tn) / total if total else np.nan
    precision = np.nan if (total_tp + total_fp) == 0 else total_tp / (total_tp + total_fp)
    recall = np.nan if (total_tp + total_fn) == 0 else total_tp / (total_tp + total_fn)
    specificity = np.nan if (total_tn + total_fp) == 0 else total_tn / (total_tn + total_fp)
    npv = np.nan if (total_tn + total_fn) == 0 else total_tn / (total_tn + total_fn)
    f1 = np.nan if pd.isna(precision) or pd.isna(recall) or (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    balanced_accuracy = np.nan if pd.isna(recall) or pd.isna(specificity) else (recall + specificity) / 2.0

    return pd.DataFrame([{
        "TP": total_tp,
        "TN": total_tn,
        "FP": total_fp,
        "FN": total_fn,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "NPV": npv,
        "F1": f1,
        "Balanced_Accuracy": balanced_accuracy,
        "Predicted_Positive": total_tp + total_fp,
        "Predicted_Negative": total_tn + total_fn,
        "Real_Positive": total_tp + total_fn,
        "Real_Negative": total_tn + total_fp,
        "Predicted_Positive_Rate": (total_tp + total_fp) / total if total else np.nan,
        "Predicted_Negative_Rate": (total_tn + total_fn) / total if total else np.nan,
    }])


In [44]:


# ============================================================================
# MAIN
# ============================================================================


In [45]:

log(
    f"--- CONFIGURATION ---\n"
    f"RUN_TAG: {RUN_TAG}\n"
    f"RAW_DATA_PATH: {RAW_DATA_PATH}\n"
    f"TARGET_COL: {TARGET_COL}\n"
    f"N_OUTER_FOLDS: {N_OUTER_FOLDS}\n"
    f"N_INNER_FOLDS: {N_INNER_FOLDS}\n"
    f"TOP_K_FEATURES: {TOP_K_FEATURES}\n"
    f"OUTER_CALIBRATION_CAP: {OUTER_CALIBRATION_CAP}\n"
    f"INNER_TRAIN_CAP: {INNER_TRAIN_CAP}\n"
    f"IHT_SAMPLING_STRATEGY: {IHT_SAMPLING_STRATEGY}\n"
    f"IHT_CV: {IHT_CV}\n"
    f"SEARCH_SIZE_CAP: {SEARCH_SIZE_CAP}\n"
    f"N_CANDIDATES_PER_INNER: {N_CANDIDATES_PER_INNER}\n"
    f"PERM_RANK_MAX_SAMPLES: {PERM_RANK_MAX_SAMPLES}\n"
    f"PERM_RANK_REPEATS: {PERM_RANK_REPEATS}\n"
    f"NUMERIC_SCALER: MinMaxScaler\n"
    f"THRESHOLD_OBJECTIVE: {THRESHOLD_OBJECTIVE}\n"
    f"MIN_THRESHOLD: {MIN_THRESHOLD}\n"
    f"MIN_SPECIFICITY_FOR_THRESHOLD: {MIN_SPECIFICITY_FOR_THRESHOLD}\n"
    f"MAX_PREDICTED_POSITIVE_RATE: {MAX_PREDICTED_POSITIVE_RATE}\n"
    f"MIN_PREDICTED_NEGATIVE_RATE: {MIN_PREDICTED_NEGATIVE_RATE}\n"
    f"USE_SIGMOID_PROBA_CALIBRATION: {USE_SIGMOID_PROBA_CALIBRATION}\n"
    f"FINAL_PROBA_CALIBRATION_SIZE: {FINAL_PROBA_CALIBRATION_SIZE}\n"
    f"FINAL_THRESHOLD_SIZE: {FINAL_THRESHOLD_SIZE}\n"
    f"OUTPUT_DIR: {OUTPUT_DIR}\n"
    f"---------------------"
)


--- CONFIGURATION ---
RUN_TAG: rf_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained_confusion
RAW_DATA_PATH: ../data/processed/dados_uteis_refreshed_v3_memorysafe.pkl
TARGET_COL: ausencia
N_OUTER_FOLDS: 10
N_INNER_FOLDS: 10
TOP_K_FEATURES: 20
OUTER_CALIBRATION_CAP: 800000
INNER_TRAIN_CAP: 250000
IHT_SAMPLING_STRATEGY: 0.5
IHT_CV: 3
SEARCH_SIZE_CAP: 60000
N_CANDIDATES_PER_INNER: 8
PERM_RANK_MAX_SAMPLES: 10000
PERM_RANK_REPEATS: 3
NUMERIC_SCALER: MinMaxScaler
THRESHOLD_OBJECTIVE: youden_j
MIN_THRESHOLD: 0.1
MIN_SPECIFICITY_FOR_THRESHOLD: 0.2
MAX_PREDICTED_POSITIVE_RATE: 0.7
MIN_PREDICTED_NEGATIVE_RATE: 0.05
USE_SIGMOID_PROBA_CALIBRATION: True
FINAL_PROBA_CALIBRATION_SIZE: 50000
FINAL_THRESHOLD_SIZE: 50000
OUTPUT_DIR: ../data/processed
---------------------


In [46]:

log("Loading raw dataset...")
df = pd.read_pickle(RAW_DATA_PATH)
if not isinstance(df, pd.DataFrame):
    raise TypeError(f"Loaded object is {type(df).__name__}, not DataFrame")
if TARGET_COL not in df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not found")

log(f"Raw DataFrame shape: {df.shape}")
y_full = df[TARGET_COL].copy()
feature_drop_cols = [c for c in DROP_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in feature_drop_cols + [TARGET_COL]]
log(f"Number of feature columns before preprocessing: {len(feature_cols)}")

outer_skf = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=SEED)

df_y_all = ensure_binary_int_labels(y_full)
print_class_distribution("FULL DATASET DISTRIBUTION", df_y_all)


Loading raw dataset...
Raw DataFrame shape: (7113798, 42)
Number of feature columns before preprocessing: 25

FULL DATASET DISTRIBUTION
-------------------------
Total: 7.113.798
Class 0: 6.069.245 (85.3165%)
Class 1: 1.044.553 (14.6835%)
Imbalance ratio (maj/min): 5.8104:1


In [47]:


outer_results = []
outer_feature_rows = []
outer_inner_candidate_rows = []
outer_inner_feature_rows = []
outer_threshold_rows = []
outer_confusion_rows = []

best_outer_fold_auc = -np.inf
best_outer_artifacts = None

for outer_fold, (cal_idx, val_idx) in enumerate(outer_skf.split(np.zeros(len(df_y_all)), df_y_all), start=1):
    outer_start = time.time()
    log("\n" + "#" * 90)
    log(f"STARTING OUTER FOLD {outer_fold}/{N_OUTER_FOLDS}")
    log("#" * 90)

    X_cal_raw_full = df.iloc[cal_idx][feature_cols].copy()
    y_cal_full = ensure_binary_int_labels(df.iloc[cal_idx][TARGET_COL], f"y_cal_outer_{outer_fold}")
    X_val_raw = df.iloc[val_idx][feature_cols].copy()
    y_val = ensure_binary_int_labels(df.iloc[val_idx][TARGET_COL], f"y_val_outer_{outer_fold}")

    # memory-safe cap for calibration side
    idx_cap_outer = make_cap_subset_indices(y_cal_full, OUTER_CALIBRATION_CAP, SEED + outer_fold)
    X_cal_raw = X_cal_raw_full.iloc[idx_cap_outer].copy()
    y_cal = y_cal_full[idx_cap_outer]

    log(f"Outer fold {outer_fold} full calibration size: {len(y_cal_full):,}".replace(",", "."))
    log(f"Outer fold {outer_fold} capped calibration size: {len(y_cal):,}".replace(",", "."))
    log(f"Outer fold {outer_fold} validation size: {len(y_val):,}".replace(",", "."))
    print_class_distribution(f"OUTER FOLD {outer_fold} - CAPPED CALIBRATION", y_cal)
    print_class_distribution(f"OUTER FOLD {outer_fold} - VALIDATION", y_val)

    num_feats, cat_feats, cat_categories = build_outer_schema(X_cal_raw)
    log(f"Outer fold {outer_fold} schema | numeric={len(num_feats)} | categorical={len(cat_feats)}")

    inner_skf = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=SEED + outer_fold)

    inner_importance_vectors = []
    inner_thresholds = []
    inner_iht_times = []
    selected_counter_inner = Counter()
    candidate_auc_accumulator = {}

    for inner_fold, (in_train_idx, in_test_idx) in enumerate(inner_skf.split(np.zeros(len(y_cal)), y_cal), start=1):
        log("\n" + "=" * 80)
        log(f"OUTER {outer_fold} | INNER {inner_fold}/{N_INNER_FOLDS}")
        log("=" * 80)

        X_in_train_raw_full = X_cal_raw.iloc[in_train_idx].copy()
        y_in_train_full = ensure_binary_int_labels(y_cal[in_train_idx], f"y_in_train_{outer_fold}_{inner_fold}")
        X_in_test_raw = X_cal_raw.iloc[in_test_idx].copy()
        y_in_test = ensure_binary_int_labels(y_cal[in_test_idx], f"y_in_test_{outer_fold}_{inner_fold}")

        idx_cap_inner = make_cap_subset_indices(y_in_train_full, INNER_TRAIN_CAP, SEED + outer_fold * 100 + inner_fold)
        X_in_train_raw = X_in_train_raw_full.iloc[idx_cap_inner].copy()
        y_in_train = y_in_train_full[idx_cap_inner]

        preprocessor_inner = build_preprocessor_from_schema(num_feats, cat_feats, cat_categories)
        preprocessor_inner.fit(X_in_train_raw)
        feature_names_inner = preprocessor_inner.get_feature_names_out().tolist()

        X_in_train = preprocessor_inner.transform(X_in_train_raw).astype(np.float32)
        X_in_test = preprocessor_inner.transform(X_in_test_raw).astype(np.float32)

        X_in_train_iht, y_in_train_iht, _, iht_time_min = apply_iht(
            X_in_train,
            y_in_train,
            SEED + outer_fold * 1000 + inner_fold,
        )
        inner_iht_times.append(iht_time_min)
        log(f"Inner fold IHT time: {iht_time_min:.2f}m")

        X_search, y_search = make_search_subset(
            X_in_train_iht,
            y_in_train_iht,
            SEARCH_SIZE_CAP,
            SEED + outer_fold * 10000 + inner_fold,
        )

        best_params_inner, best_auc_inner, df_search_inner = evaluate_candidate_grid_on_validation(
            X_search,
            y_search,
            X_in_test,
            y_in_test,
            SEED + outer_fold * 100000 + inner_fold,
        )
        df_search_inner["Outer_Fold"] = outer_fold
        df_search_inner["Inner_Fold"] = inner_fold
        outer_inner_candidate_rows.append(df_search_inner)

        for _, row in df_search_inner.iterrows():
            key = row["Candidate_Key"]
            if key not in candidate_auc_accumulator:
                candidate_auc_accumulator[key] = {
                    **{k: row[k] for k in RF_PARAM_GRID.keys()},
                    "AUCs": [],
                }
            candidate_auc_accumulator[key]["AUCs"].append(row["Validation_AUC"])

        inner_model = fit_rf_from_params(best_params_inner, SEED + outer_fold * 1000000 + inner_fold)
        inner_model.fit(X_in_train_iht, y_in_train_iht)

        y_in_test_proba = inner_model.predict_proba(X_in_test)[:, 1]
        best_thr_inner, _ = optimize_threshold_constrained(y_in_test, y_in_test_proba, THRESHOLD_OBJECTIVE)
        inner_thresholds.append(float(best_thr_inner["Threshold"]))

        importance_vector, _ = compute_permutation_importance_vector(
            model=inner_model,
            X_val=X_in_test,
            y_val=y_in_test,
            feature_names=feature_names_inner,
            seed=SEED + outer_fold * 10000000 + inner_fold,
        )
        inner_importance_vectors.append(importance_vector)

        top_idx_inner = np.argsort(-importance_vector)[:TOP_K_FEATURES]
        top_names_inner = [feature_names_inner[i] for i in top_idx_inner]
        selected_counter_inner.update(top_names_inner)

        for rank_pos, idx_feat in enumerate(top_idx_inner, start=1):
            outer_inner_feature_rows.append({
                "Outer_Fold": outer_fold,
                "Inner_Fold": inner_fold,
                "Rank_Within_Inner": rank_pos,
                "Feature_Index": int(idx_feat),
                "Feature": feature_names_inner[idx_feat],
                "Importance_Mean": float(importance_vector[idx_feat]),
                "Best_Inner_AUC": best_auc_inner,
                "Best_Inner_Threshold": float(best_thr_inner["Threshold"]),
            })

        del X_in_train_raw_full, y_in_train_full, X_in_train_raw, y_in_train
        del X_in_test_raw, y_in_test, X_in_train, X_in_test, X_in_train_iht, y_in_train_iht, X_search, y_search, inner_model
        gc.collect()

    # ------------------ Consolidation inside outer fold ------------------
    importance_matrix = np.vstack(inner_importance_vectors)
    mean_importance = importance_matrix.mean(axis=0)
    std_importance = importance_matrix.std(axis=0, ddof=1) if importance_matrix.shape[0] > 1 else np.zeros_like(mean_importance)
    feature_names_outer = feature_names_inner

    inner_feature_summary = pd.DataFrame({
        "Feature_Index": np.arange(len(feature_names_outer)),
        "Feature": feature_names_outer,
        "Mean_Inner_Importance": mean_importance,
        "Std_Inner_Importance": std_importance,
        "Selection_Frequency_Inner": [selected_counter_inner.get(f, 0) for f in feature_names_outer],
    }).sort_values(
        by=["Mean_Inner_Importance", "Selection_Frequency_Inner"],
        ascending=[False, False],
    ).reset_index(drop=True)

    consolidated_idx = inner_feature_summary.head(TOP_K_FEATURES)["Feature_Index"].to_numpy(dtype=int)
    consolidated_features = inner_feature_summary.head(TOP_K_FEATURES)["Feature"].tolist()

    candidate_summary_rows = []
    for key, d in candidate_auc_accumulator.items():
        aucs = np.array(d["AUCs"], dtype=float)
        candidate_summary_rows.append({
            "Outer_Fold": outer_fold,
            "Candidate_Key": key,
            **{k: d[k] for k in RF_PARAM_GRID.keys()},
            "Mean_Inner_AUC": float(np.nanmean(aucs)),
            "Std_Inner_AUC": float(np.nanstd(aucs, ddof=1)) if len(aucs) > 1 else 0.0,
        })
    df_candidate_summary_outer = pd.DataFrame(candidate_summary_rows).sort_values(
        by=["Mean_Inner_AUC", "Std_Inner_AUC"],
        ascending=[False, True],
    ).reset_index(drop=True)

    best_params_outer = {k: df_candidate_summary_outer.iloc[0][k] for k in RF_PARAM_GRID.keys()}
    outer_inner_candidate_rows.append(df_candidate_summary_outer.assign(Inner_Fold="summary"))

    consolidated_threshold = float(np.median(inner_thresholds))
    log(f"Outer fold {outer_fold} consolidated threshold (median inner best): {consolidated_threshold:.6f}")
    log(f"Outer fold {outer_fold} consolidated params: {best_params_outer}")
    log(f"Outer fold {outer_fold} consolidated top-{TOP_K_FEATURES} features: {consolidated_features}")

    for rank_pos, row in inner_feature_summary.head(TOP_K_FEATURES).reset_index(drop=True).iterrows():
        outer_feature_rows.append({
            "Outer_Fold": outer_fold,
            "Rank_Within_Outer": rank_pos + 1,
            "Feature_Index": int(row["Feature_Index"]),
            "Feature": row["Feature"],
            "Mean_Inner_Importance": float(row["Mean_Inner_Importance"]),
            "Std_Inner_Importance": float(row["Std_Inner_Importance"]),
            "Selection_Frequency_Inner": int(row["Selection_Frequency_Inner"]),
        })

    # ------------------ Final outer-fold training / calibration / threshold tuning ------------------
    # The final validation fold remains untouched until the final evaluation.
    # We split the capped outer calibration side into:
    #   1) final model training partition, where IHT is applied;
    #   2) natural-distribution probability calibration partition;
    #   3) natural-distribution threshold tuning partition.
    idx_final_train, idx_proba_calib, idx_threshold = split_final_train_calib_threshold_indices(
        y_cal,
        calib_size=FINAL_PROBA_CALIBRATION_SIZE,
        threshold_size=FINAL_THRESHOLD_SIZE,
        seed=SEED + outer_fold * 5000,
    )

    X_final_train_raw = X_cal_raw.iloc[idx_final_train].copy()
    y_final_train = y_cal[idx_final_train]
    X_proba_calib_raw = X_cal_raw.iloc[idx_proba_calib].copy()
    y_proba_calib = y_cal[idx_proba_calib]
    X_threshold_raw = X_cal_raw.iloc[idx_threshold].copy()
    y_threshold = y_cal[idx_threshold]

    log(f"Outer fold {outer_fold} final train size: {len(y_final_train):,}".replace(",", "."))
    log(f"Outer fold {outer_fold} probability calibration size: {len(y_proba_calib):,}".replace(",", "."))
    log(f"Outer fold {outer_fold} threshold tuning size: {len(y_threshold):,}".replace(",", "."))

    preprocessor_outer = build_preprocessor_from_schema(num_feats, cat_feats, cat_categories)
    preprocessor_outer.fit(X_final_train_raw)
    feature_names_outer_final = preprocessor_outer.get_feature_names_out().tolist()
    if feature_names_outer_final != feature_names_outer:
        raise RuntimeError(f"Feature schema mismatch in outer fold {outer_fold}.")

    X_final_train = preprocessor_outer.transform(X_final_train_raw).astype(np.float32)
    X_proba_calib = preprocessor_outer.transform(X_proba_calib_raw).astype(np.float32)
    X_threshold = preprocessor_outer.transform(X_threshold_raw).astype(np.float32)
    X_val = preprocessor_outer.transform(X_val_raw).astype(np.float32)

    X_final_train_iht, y_final_train_iht, _, outer_iht_time_min = apply_iht(
        X_final_train,
        y_final_train,
        SEED + outer_fold,
    )
    X_final_train_iht_sel = X_final_train_iht[:, consolidated_idx]
    X_proba_calib_sel = X_proba_calib[:, consolidated_idx]
    X_threshold_sel = X_threshold[:, consolidated_idx]
    X_val_sel = X_val[:, consolidated_idx]

    outer_model = fit_rf_from_params(best_params_outer, SEED + outer_fold)
    outer_model.set_params(n_jobs=2)
    outer_model.fit(X_final_train_iht_sel, y_final_train_iht)

    y_proba_calib_raw = outer_model.predict_proba(X_proba_calib_sel)[:, 1]
    y_threshold_proba_raw = outer_model.predict_proba(X_threshold_sel)[:, 1]
    y_val_proba_raw = outer_model.predict_proba(X_val_sel)[:, 1]

    calibrator = None
    calibration_method = "none_raw_rf_proba"
    if USE_SIGMOID_PROBA_CALIBRATION and len(y_proba_calib) > 0:
        calibrator = fit_sigmoid_calibrator(
            y_calib=y_proba_calib,
            proba_calib=y_proba_calib_raw,
            seed=SEED + outer_fold * 7000,
        )
        if calibrator is not None:
            calibration_method = "sigmoid_platt_on_outer_calibration"

    y_threshold_proba = apply_sigmoid_calibrator(calibrator, y_threshold_proba_raw)
    y_val_proba = apply_sigmoid_calibrator(calibrator, y_val_proba_raw)

    best_threshold_row, df_thresholds = optimize_threshold_constrained(
        y_true=y_threshold,
        y_proba=y_threshold_proba,
        objective=THRESHOLD_OBJECTIVE,
    )
    selected_threshold = float(best_threshold_row["Threshold"])

    df_thresholds.insert(0, "Outer_Fold", outer_fold)
    outer_threshold_rows.append(df_thresholds)
    threshold_path = os.path.join(OUTPUT_DIR, f"threshold_search_outerfold_{outer_fold}_{RUN_TAG}.csv")
    df_thresholds.to_csv(threshold_path, index=False)

    y_val_pred = (y_val_proba >= selected_threshold).astype(np.uint8)
    fold_metrics = calculate_metrics(y_val, y_val_pred, y_val_proba)

    confusion_row = {
        "Outer_Fold": outer_fold,
        "TN": int(fold_metrics["TN"]),
        "FP": int(fold_metrics["FP"]),
        "FN": int(fold_metrics["FN"]),
        "TP": int(fold_metrics["TP"]),
        "Real_Negative": int(fold_metrics["Real_Negative"]),
        "Real_Positive": int(fold_metrics["Real_Positive"]),
        "Predicted_Negative": int(fold_metrics["Predicted_Negative"]),
        "Predicted_Positive": int(fold_metrics["Predicted_Positive"]),
        "Predicted_Negative_Rate": float(fold_metrics["Predicted_Negative_Rate"]),
        "Predicted_Positive_Rate": float(fold_metrics["Predicted_Positive_Rate"]),
        "Selected_Threshold": selected_threshold,
        "Threshold_Objective": THRESHOLD_OBJECTIVE,
        "Threshold_Selection_Mode": best_threshold_row["Threshold_Selection_Mode"],
        "Calibration_Method": calibration_method,
        "Probability_Calibrated": bool(calibrator is not None),
    }
    outer_confusion_rows.append(confusion_row)

    log(
        f"OUTER FOLD {outer_fold} CONFUSION MATRIX [[TN, FP], [FN, TP]] = "
        f"[[{confusion_row['TN']}, {confusion_row['FP']}], "
        f"[{confusion_row['FN']}, {confusion_row['TP']}]]"
    )
    log(
        (
            f"OUTER FOLD {outer_fold} COUNTS | "
            f"RealNeg={confusion_row['Real_Negative']:,} | "
            f"RealPos={confusion_row['Real_Positive']:,} | "
            f"PredNeg={confusion_row['Predicted_Negative']:,} | "
            f"PredPos={confusion_row['Predicted_Positive']:,}"
        ).replace(",", ".")
    )

    outer_results.append({
        "Outer_Fold": outer_fold,
        "Threshold": selected_threshold,
        "Selected_Threshold": selected_threshold,
        "Inner_Median_Threshold": consolidated_threshold,
        "Threshold_Objective": THRESHOLD_OBJECTIVE,
        "Threshold_Selection_Mode": best_threshold_row["Threshold_Selection_Mode"],
        "Threshold_Valid_Candidates": int(best_threshold_row["Threshold_Valid_Candidates"]),
        "Threshold_Total_Candidates": int(best_threshold_row["Threshold_Total_Candidates"]),
        "ThresholdTune_Precision": float(best_threshold_row["Precision"]),
        "ThresholdTune_Recall": float(best_threshold_row["Recall"]),
        "ThresholdTune_Specificity": float(best_threshold_row["Specificity"]),
        "ThresholdTune_NPV": float(best_threshold_row["NPV"]) if not pd.isna(best_threshold_row["NPV"]) else np.nan,
        "ThresholdTune_F1": float(best_threshold_row["F1"]),
        "ThresholdTune_Youden_J": float(best_threshold_row["Youden_J"]),
        "ThresholdTune_Balanced_Accuracy": float(best_threshold_row["Balanced_Accuracy"]),
        "ThresholdTune_Predicted_Positive_Rate": float(best_threshold_row["Predicted_Positive_Rate"]),
        "ThresholdTune_Predicted_Negative_Rate": float(best_threshold_row["Predicted_Negative_Rate"]),
        "Calibration_Method": calibration_method,
        "Probability_Calibrated": bool(calibrator is not None),
        "N_Selected_Features": TOP_K_FEATURES,
        "Final_Train_Size": int(len(y_final_train)),
        "Proba_Calibration_Size": int(len(y_proba_calib)),
        "Threshold_Tuning_Size": int(len(y_threshold)),
        "Inner_Best_AUC_Mean": float(df_candidate_summary_outer.iloc[0]["Mean_Inner_AUC"]),
        "Inner_IHT_Time_Min_Mean": float(np.mean(inner_iht_times)),
        "Outer_IHT_Time_Min": outer_iht_time_min,
        **fold_metrics,
    })

    log(
        f"OUTER FOLD {outer_fold} COMPLETE | "
        f"Precision={fold_metrics['Precision']:.6f} | "
        f"Recall={fold_metrics['Recall']:.6f} | "
        f"Specificity={fold_metrics['Specificity']:.6f} | "
        f"NPV={fold_metrics['NPV'] if not pd.isna(fold_metrics['NPV']) else np.nan:.6f} | "
        f"F1={fold_metrics['F1']:.6f} | "
        f"ROC_AUC={fold_metrics['ROC_AUC']:.6f} | "
        f"Threshold={selected_threshold:.6f} | "
        f"PredPosRate={fold_metrics['Predicted_Positive_Rate']:.4f} | "
        f"PredNegRate={fold_metrics['Predicted_Negative_Rate']:.4f} | "
        f"ThresholdMode={best_threshold_row['Threshold_Selection_Mode']} | "
        f"Time={(time.time() - outer_start)/60.0:.1f}m"
    )

    if np.isfinite(fold_metrics["ROC_AUC"]) and fold_metrics["ROC_AUC"] > best_outer_fold_auc:
        best_outer_fold_auc = fold_metrics["ROC_AUC"]
        best_outer_artifacts = {
            "outer_fold": outer_fold,
            "model": outer_model,
            "preprocessor": preprocessor_outer,
            "calibrator": calibrator,
            "calibration_method": calibration_method,
            "selected_threshold": selected_threshold,
            "threshold_objective": THRESHOLD_OBJECTIVE,
            "threshold_row": best_threshold_row,
            "X_val_sel": X_val_sel,
            "y_val": y_val.copy(),
            "selected_feature_names": consolidated_features.copy(),
            "selected_feature_indices": consolidated_idx.copy(),
            "best_params": best_params_outer.copy(),
        }

    outer_artifact_path = os.path.join(OUTPUT_DIR, f"artifact_outerfold_{outer_fold}_{RUN_TAG}.pkl")
    joblib.dump({
        "outer_fold": outer_fold,
        "model": outer_model,
        "preprocessor": preprocessor_outer,
        "calibrator": calibrator,
        "calibration_method": calibration_method,
        "selected_threshold": selected_threshold,
        "threshold_objective": THRESHOLD_OBJECTIVE,
        "threshold_row": best_threshold_row,
        "selected_feature_names": consolidated_features.copy(),
        "selected_feature_indices": consolidated_idx.copy(),
        "best_params": best_params_outer.copy(),
    }, outer_artifact_path)

    outer_feat_path = os.path.join(OUTPUT_DIR, f"inner_feature_summary_outerfold_{outer_fold}_{RUN_TAG}.csv")
    inner_feature_summary.to_csv(outer_feat_path, index=False)

    del X_cal_raw_full, y_cal_full, X_cal_raw, y_cal, X_val_raw, y_val
    del X_final_train_raw, y_final_train, X_proba_calib_raw, y_proba_calib, X_threshold_raw, y_threshold
    del X_final_train, X_proba_calib, X_threshold, X_val
    del X_final_train_iht, y_final_train_iht, X_final_train_iht_sel
    del X_proba_calib_sel, X_threshold_sel, X_val_sel, outer_model, calibrator
    del y_proba_calib_raw, y_threshold_proba_raw, y_val_proba_raw, y_threshold_proba, y_val_proba, y_val_pred, df_thresholds
    gc.collect()




##########################################################################################
STARTING OUTER FOLD 1/10
##########################################################################################
Outer fold 1 full calibration size: 6.402.418
Outer fold 1 capped calibration size: 800.000
Outer fold 1 validation size: 711.380

OUTER FOLD 1 - CAPPED CALIBRATION
---------------------------------
Total: 800.000
Class 0: 682.532 (85.3165%)
Class 1: 117.468 (14.6835%)
Imbalance ratio (maj/min): 5.8104:1

OUTER FOLD 1 - VALIDATION
-------------------------
Total: 711.380
Class 0: 606.925 (85.3166%)
Class 1: 104.455 (14.6834%)
Imbalance ratio (maj/min): 5.8104:1
Outer fold 1 schema | numeric=15 | categorical=6

OUTER 1 | INNER 1/10
IHT distribution | before: total=250.000. class0=213.291. class1=36.709. maj/min=5.8103:1 | after: total=110.127. class0=73.418. class1=36.709. maj/min=2.0000:1 | sampling_strategy=0.5
Inner fold IHT time: 0.14m

OUTER 1 | INNER 2/10
IHT distribution | be

In [48]:

# ------------------ Save outputs ------------------
df_outer = pd.DataFrame(outer_results)
results_path = os.path.join(OUTPUT_DIR, f"outer_fold_results_{RUN_TAG}.csv")
df_outer.to_csv(results_path, index=False)

df_confusion = pd.DataFrame(outer_confusion_rows)
confusion_path = os.path.join(OUTPUT_DIR, f"confusion_matrices_{RUN_TAG}.csv")
df_confusion.to_csv(confusion_path, index=False)

summary_df = summarize_outer_results(df_outer)
summary_path = os.path.join(OUTPUT_DIR, f"summary_mean_std_{RUN_TAG}.csv")
summary_df.to_csv(summary_path, index=False)

pooled_summary_df = summarize_pooled_confusion(df_outer)
pooled_summary_path = os.path.join(OUTPUT_DIR, f"pooled_confusion_summary_{RUN_TAG}.csv")
pooled_summary_df.to_csv(pooled_summary_path, index=False)

if outer_threshold_rows:
    threshold_all_df = pd.concat(outer_threshold_rows, ignore_index=True)
else:
    threshold_all_df = pd.DataFrame()
threshold_all_path = os.path.join(OUTPUT_DIR, f"threshold_search_all_outerfolds_{RUN_TAG}.csv")
threshold_all_df.to_csv(threshold_all_path, index=False)

outer_feature_df = pd.DataFrame(outer_feature_rows)
outer_feature_path = os.path.join(OUTPUT_DIR, f"outer_consolidated_top{TOP_K_FEATURES}_{RUN_TAG}.csv")
outer_feature_df.to_csv(outer_feature_path, index=False)

inner_feature_df = pd.DataFrame(outer_inner_feature_rows)
inner_feature_path = os.path.join(OUTPUT_DIR, f"inner_selected_features_{RUN_TAG}.csv")
inner_feature_df.to_csv(inner_feature_path, index=False)

candidate_diag_df = pd.concat(outer_inner_candidate_rows, ignore_index=True)
candidate_diag_path = os.path.join(OUTPUT_DIR, f"candidate_diagnostics_{RUN_TAG}.csv")
candidate_diag_df.to_csv(candidate_diag_path, index=False)

global_feature_summary = (
    outer_feature_df.groupby("Feature")
    .agg(
        Outer_Selection_Frequency=("Feature", "count"),
        Mean_Outer_Rank=("Rank_Within_Outer", "mean"),
        Mean_Outer_Importance=("Mean_Inner_Importance", "mean"),
        Std_Outer_Importance=("Mean_Inner_Importance", "std"),
    )
    .reset_index()
)
global_feature_summary = global_feature_summary.sort_values(
    by=["Outer_Selection_Frequency", "Mean_Outer_Importance", "Mean_Outer_Rank"],
    ascending=[False, False, True],
).reset_index(drop=True)

global_feature_summary_path = os.path.join(OUTPUT_DIR, f"global_feature_summary_{RUN_TAG}.csv")
global_feature_summary.to_csv(global_feature_summary_path, index=False)

global_top20_df = global_feature_summary.head(TOP_K_FEATURES).copy()
global_top20_path = os.path.join(OUTPUT_DIR, f"global_top{TOP_K_FEATURES}_{RUN_TAG}.csv")
global_top20_df.to_csv(global_top20_path, index=False)

if best_outer_artifacts is not None:
    best_model_path = os.path.join(OUTPUT_DIR, f"best_outer_model_{RUN_TAG}.pkl")
    best_artifacts_path = os.path.join(OUTPUT_DIR, f"best_outer_artifacts_{RUN_TAG}.pkl")
    joblib.dump(best_outer_artifacts["model"], best_model_path)
    joblib.dump(best_outer_artifacts, best_artifacts_path)

    best_meta = {
        "best_outer_fold": int(best_outer_artifacts["outer_fold"]),
        "best_outer_fold_auc": float(best_outer_fold_auc),
        "best_outer_selected_features": best_outer_artifacts["selected_feature_names"],
        "selected_threshold": float(best_outer_artifacts["selected_threshold"]),
        "threshold_objective": best_outer_artifacts["threshold_objective"],
        "calibration_method": best_outer_artifacts["calibration_method"],
        "best_params": best_outer_artifacts["best_params"],
    }
    with open(os.path.join(OUTPUT_DIR, f"best_outer_model_metadata_{RUN_TAG}.json"), "w", encoding="utf-8") as f:
        json.dump(best_meta, f, ensure_ascii=False, indent=2, default=str)

    if RUN_OUTER_BEST_FOLD_PERMUTATION:
        X_perm, y_perm = make_perm_subset(
            best_outer_artifacts["X_val_sel"],
            best_outer_artifacts["y_val"],
            OUTER_PERM_MAX_SAMPLES,
            SEED,
        )
        perm_results = permutation_importance(
            estimator=best_outer_artifacts["model"],
            X=X_perm,
            y=y_perm,
            scoring=roc_auc_proba_scorer,
            n_repeats=OUTER_PERM_REPEATS,
            random_state=SEED,
            n_jobs=1,
            max_samples=1.0,
        )
        df_perm_best_outer = pd.DataFrame({
            "Feature": best_outer_artifacts["selected_feature_names"],
            "Importance_Mean": perm_results.importances_mean,
            "Importance_Std": perm_results.importances_std,
        }).sort_values(by="Importance_Mean", ascending=False)
        df_perm_best_outer.to_csv(
            os.path.join(OUTPUT_DIR, f"best_outer_permutation_{RUN_TAG}.csv"),
            index=False,
        )


In [49]:

config = {
    "RUN_TAG": RUN_TAG,
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "TARGET_COL": TARGET_COL,
    "N_OUTER_FOLDS": N_OUTER_FOLDS,
    "N_INNER_FOLDS": N_INNER_FOLDS,
    "TOP_K_FEATURES": TOP_K_FEATURES,
    "OUTER_CALIBRATION_CAP": OUTER_CALIBRATION_CAP,
    "INNER_TRAIN_CAP": INNER_TRAIN_CAP,
    "IHT_SAMPLING_STRATEGY": IHT_SAMPLING_STRATEGY,
    "IHT_CV": IHT_CV,
    "IHT_HARDNESS_MAX_ITER": IHT_HARDNESS_MAX_ITER,
    "IHT_HARDNESS_TOL": IHT_HARDNESS_TOL,
    "IHT_HARDNESS_ALPHA": IHT_HARDNESS_ALPHA,
    "IHT_HARDNESS_CLASS_WEIGHT": IHT_HARDNESS_CLASS_WEIGHT,
    "SEARCH_SIZE_CAP": SEARCH_SIZE_CAP,
    "N_CANDIDATES_PER_INNER": N_CANDIDATES_PER_INNER,
    "PERM_RANK_MAX_SAMPLES": PERM_RANK_MAX_SAMPLES,
    "PERM_RANK_REPEATS": PERM_RANK_REPEATS,
    "NUMERIC_SCALER": "MinMaxScaler",
    "THRESHOLD_OBJECTIVE": THRESHOLD_OBJECTIVE,
    "MANUAL_THRESHOLDS": MANUAL_THRESHOLDS,
    "MIN_THRESHOLD": MIN_THRESHOLD,
    "MIN_SPECIFICITY_FOR_THRESHOLD": MIN_SPECIFICITY_FOR_THRESHOLD,
    "MIN_PRECISION_MARGIN_OVER_PREVALENCE": MIN_PRECISION_MARGIN_OVER_PREVALENCE,
    "MAX_PREDICTED_POSITIVE_RATE": MAX_PREDICTED_POSITIVE_RATE,
    "MIN_PREDICTED_NEGATIVE_RATE": MIN_PREDICTED_NEGATIVE_RATE,
    "USE_SIGMOID_PROBA_CALIBRATION": USE_SIGMOID_PROBA_CALIBRATION,
    "FINAL_PROBA_CALIBRATION_SIZE": FINAL_PROBA_CALIBRATION_SIZE,
    "FINAL_THRESHOLD_SIZE": FINAL_THRESHOLD_SIZE,
    "RF_PARAM_GRID": RF_PARAM_GRID,
}
with open(os.path.join(OUTPUT_DIR, f"config_{RUN_TAG}.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2, default=str)


In [50]:

log("\n" + "#" * 90)
log("NESTED-CV RANDOM FOREST WORKFLOW COMPLETE")
log("#" * 90)
log(f"Outer-fold results saved: {results_path}")
log(f"Confusion matrices saved: {confusion_path}")
log(f"Mean/std summary saved: {summary_path}")
log(f"Pooled confusion summary saved: {pooled_summary_path}")
log(f"Threshold search all folds saved: {threshold_all_path}")
log(f"Outer consolidated top-{TOP_K_FEATURES} saved: {outer_feature_path}")
log(f"Global feature summary saved: {global_feature_summary_path}")
log(f"Global top-{TOP_K_FEATURES} saved: {global_top20_path}")
log(f"Candidate diagnostics saved: {candidate_diag_path}")



##########################################################################################
NESTED-CV RANDOM FOREST WORKFLOW COMPLETE
##########################################################################################
Outer-fold results saved: ../data/processed/outer_fold_results_rf_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained_confusion.csv
Confusion matrices saved: ../data/processed/confusion_matrices_rf_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained_confusion.csv
Mean/std summary saved: ../data/processed/summary_mean_std_rf_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained_confusion.csv
Pooled confusion summary saved: ../data/processed/pooled_confusion_summary_rf_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained_confusion.csv
Threshold search all folds saved: ../data/processed/threshold_search_all_outerfolds_rf_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained_confusion.csv
Outer consolidated to

In [51]:

log("\nOUTER FOLD CONFUSION MATRICES")
log(str(df_confusion[[
    "Outer_Fold", "TN", "FP", "FN", "TP",
    "Real_Negative", "Real_Positive",
    "Predicted_Negative", "Predicted_Positive",
    "Predicted_Negative_Rate", "Predicted_Positive_Rate",
    "Selected_Threshold", "Threshold_Selection_Mode",
]]))



OUTER FOLD CONFUSION MATRICES
   Outer_Fold      TN      FP     FN     TP  Real_Negative  Real_Positive  \
0           1  346970  259955  34134  70321         606925         104455   
1           2  392982  213943  42005  62450         606925         104455   
2           3  362151  244774  36502  67953         606925         104455   
3           4  361957  244968  36862  67593         606925         104455   
4           5  360511  246414  36388  68067         606925         104455   
5           6  379825  227099  39726  64730         606924         104456   
6           7  361909  245015  36793  67663         606924         104456   
7           8  388428  218496  40626  63830         606924         104456   
8           9  373237  233687  38445  66010         606924         104455   
9          10  374857  232067  38992  65463         606924         104455   

   Predicted_Negative  Predicted_Positive  Predicted_Negative_Rate  \
0              381104              330276          

In [52]:

log("\nOUTER FOLD METRICS")
metric_cols_to_print = [
    "Outer_Fold", "Precision", "Recall", "Specificity", "NPV", "F1", "ROC_AUC",
    "Threshold", "Predicted_Positive_Rate", "Predicted_Negative_Rate",
    "TP", "TN", "FP", "FN",
]
log(str(df_outer[metric_cols_to_print]))



OUTER FOLD METRICS
   Outer_Fold  Precision    Recall  Specificity       NPV        F1   ROC_AUC  \
0           1   0.212916  0.673218     0.571685  0.910434  0.323515  0.672002   
1           2   0.225946  0.597865     0.647497  0.903434  0.327952  0.671755   
2           3   0.217292  0.650548     0.596698  0.908437  0.325771  0.672125   
3           4   0.216255  0.647102     0.596378  0.907572  0.324175  0.669234   
4           5   0.216442  0.651639     0.593996  0.908319  0.324952  0.670940   
5           6   0.221808  0.619687     0.625820  0.905313  0.326684  0.671457   
6           7   0.216398  0.647766     0.596300  0.907718  0.324419  0.667498   
7           8   0.226086  0.611071     0.639994  0.905313  0.330057  0.674088   
8           9   0.220256  0.631947     0.614965  0.906615  0.326659  0.670705   
9          10   0.220022  0.626710     0.617634  0.905782  0.325699  0.669118   

   Threshold  Predicted_Positive_Rate  Predicted_Negative_Rate     TP      TN  \
0   0.1

In [53]:

log("\nMEAN / STD SUMMARY")
log(str(summary_df))



MEAN / STD SUMMARY
                     Metric      Mean       Std       Min       Max  \
0                  Accuracy  0.613864  0.016648  0.586594  0.640209   
1                 Precision  0.219342  0.004328  0.212916  0.226086   
2                    Recall  0.635755  0.022477  0.597865  0.673218   
3               Specificity  0.610097  0.023365  0.571685  0.647497   
4                       NPV  0.906894  0.002010  0.903434  0.910434   
5                        F1  0.325988  0.001956  0.323515  0.330057   
6                   ROC_AUC  0.670892  0.001870  0.667498  0.674088   
7                 Threshold  0.140537  0.005777  0.130286  0.151188   
8        Selected_Threshold  0.140537  0.005777  0.130286  0.151188   
9    Inner_Median_Threshold  0.759947  0.023702  0.725000  0.806133   
10  Predicted_Positive_Rate  0.426003  0.023224  0.388531  0.464275   
11  Predicted_Negative_Rate  0.573997  0.023224  0.535725  0.611469   
12      Inner_Best_AUC_Mean  0.675347  0.001958  0.671854

In [54]:

log("\nPOOLED CONFUSION SUMMARY")
log(str(pooled_summary_df.T))



POOLED CONFUSION SUMMARY
                                    0
TP                       6.640800e+05
TN                       3.702827e+06
FP                       2.366418e+06
FN                       3.804730e+05
Accuracy                 6.138644e-01
Precision                2.191323e-01
Recall                   6.357552e-01
Specificity              6.100968e-01
NPV                      9.068222e-01
F1                       3.259248e-01
Balanced_Accuracy        6.229260e-01
Predicted_Positive       3.030498e+06
Predicted_Negative       4.083300e+06
Real_Positive            1.044553e+06
Real_Negative            6.069245e+06
Predicted_Positive_Rate  4.260028e-01
Predicted_Negative_Rate  5.739972e-01


In [55]:
log("\nGLOBAL TOP FEATURES")
log(str(global_top20_df))


GLOBAL TOP FEATURES
                         Feature  Outer_Selection_Frequency  Mean_Outer_Rank  \
0                   tempo_espera                         10         1.000000   
1   percentual_faltas_anteriores                         10         2.000000   
2                  idade_usuario                         10         3.000000   
3     contagem_faltas_anteriores                         10         4.100000   
4         mesma_unidade_anterior                         10         5.000000   
5               cbo_espec_225265                         10         5.900000   
6               cbo_espec_223710                         10         8.200000   
7               cbo_espec_223208                         10         8.900000   
8                 tip_marcacao_0                         10         9.900000   
9                 tip_marcacao_2                         10         9.800000   
10                   distance_km                         10         9.800000   
11           raca_c